In [1]:
import numpy as np
import matplotlib.pyplot as plt

def plot_reliability_diagram(preds, confs, labels, n_bins=15, title=None, save_dir=None):
    
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(confs, bins) - 1

    bin_acc = []
    bin_confidences = []
    for i in range(n_bins):
        in_bin = bin_indices == i

        if np.sum(in_bin) > 0:
            accuracy = np.mean(labels[in_bin] == preds[in_bin])
            mean_confidence = np.mean(confs[in_bin])
        else:
            accuracy = 0
            mean_confidence = 0
        bin_acc.append(accuracy)
        bin_confidences.append(mean_confidence)

    bin_acc = np.array(bin_acc)
    bin_confidences = np.array(bin_confidences)

    weights = np.histogram(confs, bins)[0] / len(confs)
    ece = np.sum(weights * np.abs(bin_confidences - bin_acc))

    # plot
    delta = 1.0 / n_bins
    x = np.arange(0, 1, delta)
    mid = np.linspace(delta / 2, 1 - delta / 2, n_bins)
    error = np.abs(np.subtract(mid, bin_acc))

    plt.rcParams["font.family"] = "serif"
    
    # Size and axis limits
    plt.figure(figsize=(6.7, 6))
    plt.xlim(0, 1)
    plt.ylim(0, 1)

    # Plot grid
    plt.grid(color='tab:grey', linestyle=(0, (1, 5)), linewidth=1, zorder=0)

    # Plot bars and identity line
    plt.bar(x, bin_acc, color='#6B9EFF', width=delta, align='edge', edgecolor='k', label='Outputs', zorder=5)
    plt.bar(x, error, bottom=np.minimum(bin_acc, mid), color='mistyrose', alpha=0.5, width=delta, align='edge', 
            edgecolor='r', hatch='/', label='Gap', zorder=10)
    ident = [0.0, 1.0]
    plt.plot(ident, ident, linestyle='--', color='tab:grey', zorder=15)

    # Labels and legend
    plt.ylabel('Accuracy', fontsize=24)  # Increased font size
    plt.xlabel('Confidence', fontsize=24)  # Increased font size
    plt.legend(loc='upper left', framealpha=1.0, fontsize=22)  # Enlarged legend

    # Increased size of x and y axis ticks
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)


    # ECE text in bottom-right corner with higher zorder to float above bars
    plt.text(0.47, 0.065, f'ECE: {ece * 100:.2f}%',  # Changed to bottom-right position
             transform=plt.gca().transAxes,
             bbox=dict(boxstyle='round, pad=0.5', facecolor='#FFDAB9', edgecolor='#D2691E'),  # Changed color
             fontsize=28, color='darkblue', zorder=20)  # Ensure text floats on top of bars



    if title is not None:
        plt.title(title, fontsize=36)  # Increased title font size


    # legend = plt.legend()  # 获取图例对象
    # legend.set_visible(False)  # 隐藏图例
    # plt.ylabel('')  # Increased font size

    
    plt.tight_layout()
    # Save ECE figure
    if save_dir:
        plt.savefig(save_dir)

    return plt

# ================== Test Code ==================
# Simulate predictions, confidences, and labels
np.random.seed(42)
n_samples = 500
confs = np.random.beta(2, 5, size=n_samples)  # Skewed confidence distribution
preds = np.random.randint(0, 2, size=n_samples)
labels = (preds == np.random.randint(0, 2, size=n_samples)).astype(int)

# Plot the diagram
plt = plot_reliability_diagram(preds, confs, labels, n_bins=10, title='Calibration', save_dir='reliability_diagram.png')
plt.show()

ModuleNotFoundError: No module named 'matplotlib'